# 03 SFT response mask 与 DPO loss
本 notebook 用真实 tokenizer 和训练模块验证两个最容易写错的目标函数。

In [ ]:
from pathlib import Path
import sys, torch
ROOT = Path.cwd()
if not (ROOT / 'model').exists(): ROOT = ROOT.parents[2]
sys.path.insert(0, str(ROOT))
from tokenizer.tokenizer_utils import MiniLLMTokenizer
from train.dpo import DPODataset, dpo_loss
tok = MiniLLMTokenizer(str(ROOT / 'tokenizer/minillm_tokenizer.json'))

DPO 数据集和 SFT 数据集都先移位，再把 prompt target 标成 -100。下面直接调用生产代码检查。

In [ ]:
ds = DPODataset(str(ROOT / 'data/instruction_dpo_v2/train.jsonl'), tok, max_length=256)
chosen_ids, chosen_labels, _, _ = ds[0]
valid = chosen_labels != -100
print('sequence tokens:', len(chosen_ids))
print('supervised tokens:', int(valid.sum()))
print('first supervised target:', tok.decode([int(chosen_labels[valid][0])]))
assert valid.any() and (~valid).any()

若策略相对 reference 更偏好 chosen，DPO loss 应小于随机边界 $-\log \sigma(0)=\log 2$。

In [ ]:
ref_c = torch.tensor([-2.0])
ref_r = torch.tensor([-2.0])
policy_c = torch.tensor([-1.5])
policy_r = torch.tensor([-2.5])
good_loss = dpo_loss(policy_c, policy_r, ref_c, ref_r, beta=1.0)
neutral_loss = dpo_loss(ref_c, ref_r, ref_c, ref_r, beta=1.0)
print('improved preference loss:', good_loss.item())
print('neutral loss (log 2):   ', neutral_loss.item())
assert good_loss < neutral_loss

In [ ]:
for margin in [-2, -1, 0, 1, 2]:
    loss = dpo_loss(torch.tensor([float(margin)]), torch.tensor([0.]), torch.tensor([0.]), torch.tensor([0.]), beta=1.)
    print(f'margin={margin:+d}, loss={loss.item():.4f}')